<a href="https://colab.research.google.com/github/Khushibung05/NLP/blob/main/positional_encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###positional encoding

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, MultiHeadAttention

####Input Sentence

In [ ]:
import numpy as np
sentence=["I love deep learning"]
print(sentence)

['I love deep learning']


####Tokenization

In [ ]:
vectorizer=TextVectorization(output_mode="int",output_sequence_length=4)
vectorizer.adapt(sentence)
tokens=vectorizer(sentence)

print("Vocabulary:")
print(vectorizer.get_vocabulary())
print("Tokens:")
print(tokens.numpy())


Vocabulary:
['', '[UNK]', np.str_('love'), np.str_('learning'), np.str_('i'), np.str_('deep')]
Tokens:
[[4 2 5 3]]


####Word Embeddings

In [ ]:
embedding_dim=8
embedding_layer=Embedding(input_dim=len(vectorizer.get_vocabulary()),output_dim=embedding_dim)
word_embeddings=embedding_layer(tokens)
print("Word Embeddings:")
print(word_embeddings.numpy())


Word Embeddings:
[[[-0.00349903 -0.03774176  0.01011395 -0.01831728 -0.02562647
    0.01430679  0.0486244  -0.04064365]
  [ 0.02622353  0.01405748 -0.01188434 -0.00771729 -0.04620099
   -0.04329598 -0.04690861  0.02500382]
  [-0.0106272  -0.03923129 -0.00641782  0.0444762  -0.0037635
   -0.01089407  0.03353849 -0.02066731]
  [-0.03659139  0.00758518  0.02270829 -0.02718086 -0.00206177
   -0.04690119  0.02487514 -0.01308512]]]


####Positional Encoding Function

In [ ]:
def positional_encoding(max_position,d_model):

  positions=np.arange(max_position)[:,np.newaxis]
  dimensions=np.arange(d_model)[np.newaxis,:]
  angle_rates=1/np.power(10000,(2*(dimensions//2))/np.float32(d_model))
  angle_rads=positions*angle_rates
  PE=np.zeros((max_position,d_model))
  PE[:,0::2]=np.sin(angle_rads[:,0::2])
  PE[:,1::2]=np.cos(angle_rads[:,1::2])
  return tf.cast(PE,dtype=tf.float32)

####Generate Positional Encoding

In [ ]:
PE=positional_encoding(4,embedding_dim)
print("Positional Encoding:")
print(PE.numpy())

Positional Encoding:
[[ 0.0000000e+00  1.0000000e+00  0.0000000e+00  1.0000000e+00
   0.0000000e+00  1.0000000e+00  0.0000000e+00  1.0000000e+00]
 [ 8.4147096e-01  5.4030228e-01  9.9833414e-02  9.9500418e-01
   9.9998331e-03  9.9994999e-01  9.9999981e-04  9.9999952e-01]
 [ 9.0929741e-01 -4.1614684e-01  1.9866933e-01  9.8006660e-01
   1.9998666e-02  9.9980003e-01  1.9999987e-03  9.9999797e-01]
 [ 1.4112000e-01 -9.8999250e-01  2.9552022e-01  9.5533651e-01
   2.9995501e-02  9.9955004e-01  2.9999956e-03  9.9999553e-01]]


###Add Positional Encoding

In [ ]:
position_aware_embeddings=word_embeddings+PE[tf.newaxis,:]
print("Position-aware Embeddings:")
print(position_aware_embeddings.numpy())

Position-aware Embeddings:
[[[-0.00349903  0.9622582   0.01011395  0.9816827  -0.02562647
    1.0143068   0.0486244   0.95935637]
  [ 0.8676945   0.55435973  0.08794907  0.98728687 -0.03620116
    0.956654   -0.04590861  1.0250033 ]
  [ 0.8986702  -0.45537814  0.1922515   1.0245428   0.01623517
    0.98890597  0.03553849  0.97933066]
  [ 0.10452861 -0.98240733  0.3182285   0.92815566  0.02793373
    0.9526489   0.02787513  0.9869104 ]]]


####Multi-Head Attention

In [ ]:
attention_layer=MultiHeadAttention(
    num_heads=2,
    key_dim=embedding_dim
)

####Apply self-Attention

In [ ]:
attention_output=attention_layer(
    query=position_aware_embeddings,
    value=position_aware_embeddings,
    key=position_aware_embeddings,
)
print(attention_output.shape)
print("Contextulized Embeddings:")
print(attention_output.numpy())

(1, 4, 8)
Contextulized Embeddings:
[[[ 0.45965534 -0.14375955 -0.11247128  0.04943637  0.45942843
   -0.29046795  0.38458127  0.03913592]
  [ 0.4558742  -0.14320737 -0.10848828  0.04862841  0.45546827
   -0.28988972  0.38822424  0.03993595]
  [ 0.4510886  -0.14184147 -0.10387745  0.04723673  0.45036978
   -0.28956792  0.3921917   0.0412361 ]
  [ 0.45093024 -0.1411176  -0.10414466  0.04675968  0.45014685
   -0.28994715  0.39167348  0.0415899 ]]]
